# 04 - Modèle avec contexte

Ce notebook entraîne un modèle NCF qui intègre des features contextuelles extraites de MovieLens 100K.

L'objectif est de comparer une architecture baseline sans contexte avec un modèle qui utilise l'heure, le jour de la semaine, et des features de session.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

sys.path.append(str((Path('..') / 'src').resolve()))
from data_utils import load_movielens_100k, encode_ids
from context_features import extract_all_context_features
from models import NCFContext
from metrics import hit_rate, ndcg, mrr

print('Python:', sys.version.split()[0])
print('pandas :', pd.__version__)
print('numpy :', np.__version__)
print('torch :', torch.__version__)


Python: 3.12.13
pandas : 3.0.3
numpy : 2.4.6
torch : 2.12.0+cu130


## Chargement et extraction des features de contexte

Nous utilisons MovieLens 100K en chargeant le fichier brut et en extrayant les features temporelles et de session.


In [2]:
NOTEBOOK_ROOT = Path('.')
RAW_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'raw' / 'movielens' / 'ml-100k').resolve()
print('RAW_ROOT =', RAW_ROOT)

df = load_movielens_100k(RAW_ROOT)
print('raw shape:', df.shape)

df = encode_ids(df)
df = extract_all_context_features(df)

print('processed shape:', df.shape)
print(df.columns.tolist())
print(df[['user_id_encoded', 'item_id_encoded', 'hour_of_day', 'dow_sin', 'session_length', 'session_position_norm']].head())


RAW_ROOT = /home/mrtds/Documents/my_projects/context-aware-recsys/data/raw/movielens/ml-100k
raw shape: (100000, 4)
processed shape: (100000, 19)
['user_id', 'item_id', 'rating', 'timestamp', 'user_id_encoded', 'item_id_encoded', 'hour_of_day', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'time_since_last_interaction', 'session_id', 'session_length', 'session_position', 'session_position_norm', 'time_since_last_interaction_log']
   user_id_encoded  item_id_encoded  hour_of_day  dow_sin  session_length  \
0                0              167           21      0.0              20   
1                0              171           21      0.0              20   
2                0              164           21      0.0              20   
3                0              155           21      0.0              20   
4                0              195           22      0.0              20   

   session_position_norm  
0               0.000000  
1               0.05

## Chargement et extraction des features de contexte - RetailRocket

In [3]:
retailrocket_path = '/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/retailrocket/interactions.parquet'
df_retailrocket = pd.read_parquet(retailrocket_path)
print('retailrocket raw shape:', df_retailrocket.shape)

df_retailrocket = extract_all_context_features(df_retailrocket)
print('retailrocket processed shape:', df_retailrocket.shape)
print(df_retailrocket.columns.tolist())
print(df_retailrocket[['user_id_encoded', 'item_id_encoded', 'hour_of_day', 'dow_sin', 'session_length', 'session_position_norm']].head())

retailrocket raw shape: (781519, 21)
retailrocket processed shape: (781519, 21)
['timestamp', 'user_id', 'event', 'item_id', 'transactionid', 'user_id_encoded', 'item_id_encoded', 'hour_of_day', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_desktop_proxy', 'time_since_last_interaction', 'session_id', 'session_length', 'session_position', 'session_position_norm', 'time_since_last_interaction_log']
   user_id_encoded  item_id_encoded  hour_of_day   dow_sin  session_length  \
0                0            25197           17 -0.433884               8   
1                0            25197           17 -0.433884               8   
2                0            20203           17 -0.433884               8   
3                0            16772           18 -0.433884               8   
4                0            26527           18 -0.433884               8   

   session_position_norm  
0               0.000000  
1               0.142857  
2               0

## Chargement et extraction des features de contexte - MovieLens 1M

In [4]:
ml_1m_path = '/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_1m/interactions.parquet'
df_ml_1m = pd.read_parquet(ml_1m_path)
print('ml-1m raw shape:', df_ml_1m.shape)

df_ml_1m = extract_all_context_features(df_ml_1m)
print('ml-1m processed shape:', df_ml_1m.shape)
print(df_ml_1m.columns.tolist())
print(df_ml_1m[['user_id_encoded', 'item_id_encoded', 'hour_of_day', 'dow_sin', 'session_length', 'session_position_norm']].head())

ml-1m raw shape: (999611, 19)
ml-1m processed shape: (999611, 19)
['user_id', 'item_id', 'rating', 'timestamp', 'user_id_encoded', 'item_id_encoded', 'hour_of_day', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'time_since_last_interaction', 'session_id', 'session_length', 'session_position', 'session_position_norm', 'time_since_last_interaction_log']
   user_id_encoded  item_id_encoded  hour_of_day   dow_sin  session_length  \
0                0             2757           22 -0.781831              40   
1                0              861           22 -0.781831              40   
2                0             1444           22 -0.781831              40   
3                0             1068           22 -0.781831              40   
4                0             1978           22 -0.781831              40   

   session_position_norm  
0               0.000000  
1               0.025641  
2               0.051282  
3               0.076923  
4            

### Features contextuelles utilisées

Nous utilisons un ensemble de features continues extraites pour représenter l'heure, le jour de la semaine, et le comportement de session.


In [5]:
context_cols = [
    'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos',
    'is_weekend',
    'time_since_last_interaction_log',
    'session_length',
    'session_position_norm'
]
print('Context columns:', context_cols)
print(df[context_cols].head())


Context columns: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'time_since_last_interaction_log', 'session_length', 'session_position_norm']
   hour_sin  hour_cos  dow_sin  dow_cos  is_weekend  \
0 -0.707107  0.707107      0.0      1.0           0   
1 -0.707107  0.707107      0.0      1.0           0   
2 -0.707107  0.707107      0.0      1.0           0   
3 -0.707107  0.707107      0.0      1.0           0   
4 -0.500000  0.866025      0.0      1.0           0   

   time_since_last_interaction_log  session_length  session_position_norm  
0                         0.000000              20               0.000000  
1                         0.000000              20               0.052632  
2                         3.713572              20               0.105263  
3                         3.663562              20               0.157895  
4                         4.804021              20               0.210526  


### Features contextuelles - RetailRocket

In [6]:
print('RetailRocket Context columns:', context_cols)
print(df_retailrocket[context_cols].head())

RetailRocket Context columns: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'time_since_last_interaction_log', 'session_length', 'session_position_norm']
   hour_sin      hour_cos   dow_sin   dow_cos  is_weekend  \
0 -0.965926 -2.588190e-01 -0.433884 -0.900969           0   
1 -0.965926 -2.588190e-01 -0.433884 -0.900969           0   
2 -0.965926 -2.588190e-01 -0.433884 -0.900969           0   
3 -1.000000 -1.836970e-16 -0.433884 -0.900969           0   
4 -1.000000 -1.836970e-16 -0.433884 -0.900969           0   

   time_since_last_interaction_log  session_length  session_position_norm  
0                         0.000000               8               0.000000  
1                         4.702506               8               0.142857  
2                         5.297687               8               0.285714  
3                         5.550072               8               0.428571  
4                         6.081646               8               0.571429  


### Features contextuelles - MovieLens 1M

In [7]:
print('ML-1M Context columns:', context_cols)
print(df_ml_1m[context_cols].head())

ML-1M Context columns: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'time_since_last_interaction_log', 'session_length', 'session_position_norm']
   hour_sin  hour_cos   dow_sin  dow_cos  is_weekend  \
0      -0.5  0.866025 -0.781831  0.62349           1   
1      -0.5  0.866025 -0.781831  0.62349           1   
2      -0.5  0.866025 -0.781831  0.62349           1   
3      -0.5  0.866025 -0.781831  0.62349           1   
4      -0.5  0.866025 -0.781831  0.62349           1   

   time_since_last_interaction_log  session_length  session_position_norm  
0                         0.000000              40               0.000000  
1                         3.610918              40               0.025641  
2                         0.000000              40               0.051282  
3                         0.000000              40               0.076923  
4                         3.891820              40               0.102564  


## Préparation des labels

Nous convertissons les notes en labels binaires pour entraîner un modèle de ranking simple.


In [8]:
df['label'] = (df['rating'] >= 4).astype(int)
print(df['label'].value_counts(normalize=True))

n_users = df['user_id_encoded'].nunique()
n_items = df['item_id_encoded'].nunique()
context_dim = len(context_cols)

print('Users:', n_users, 'Items:', n_items, 'Context dim:', context_dim)


label
1    0.55375
0    0.44625
Name: proportion, dtype: float64
Users: 943 Items: 1682 Context dim: 8


## Préparation des labels - RetailRocket

In [9]:
if 'rating' in df_retailrocket.columns:
    df_retailrocket['label'] = (df_retailrocket['rating'] >= 4).astype(int)
else:
    df_retailrocket['label'] = 1

print(df_retailrocket['label'].value_counts(normalize=True))

n_users_retail = df_retailrocket['user_id_encoded'].nunique()
n_items_retail = df_retailrocket['item_id_encoded'].nunique()
context_dim_retail = len(context_cols)

print('RetailRocket Users:', n_users_retail, 'Items:', n_items_retail, 'Context dim:', context_dim_retail)

label
1    1.0
Name: proportion, dtype: float64
RetailRocket Users: 65968 Items: 36126 Context dim: 8


## Préparation des labels - MovieLens 1M

In [10]:
df_ml_1m['label'] = (df_ml_1m['rating'] >= 4).astype(int)
print(df_ml_1m['label'].value_counts(normalize=True))

n_users_1m = df_ml_1m['user_id_encoded'].nunique()
n_items_1m = df_ml_1m['item_id_encoded'].nunique()
context_dim_1m = len(context_cols)

print('ML-1M Users:', n_users_1m, 'Items:', n_items_1m, 'Context dim:', context_dim_1m)

label
1    0.57529
0    0.42471
Name: proportion, dtype: float64
ML-1M Users: 6040 Items: 3416 Context dim: 8


## Split train/test

Nous utilisons un split aléatoire en 80/20. On pourra remplacer par un split temporel spécifique plus tard.


In [11]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print('train shape:', train_df.shape)
print('test shape:', test_df.shape)


train shape: (80000, 20)
test shape: (20000, 20)


## Split train/test - MovieLens 1M

In [12]:
def safe_train_test_split(df, label_col='label'):
    if label_col in df.columns and df[label_col].nunique() > 1:
        stratify = df[label_col]
    else:
        stratify = None
    return train_test_split(df, test_size=0.2, random_state=42, stratify=stratify)

train_df_ml_1m, test_df_ml_1m = safe_train_test_split(df_ml_1m)
print('ML-1M train shape:', train_df_ml_1m.shape)
print('ML-1M test shape:', test_df_ml_1m.shape)

ML-1M train shape: (799688, 20)
ML-1M test shape: (199923, 20)


## Split train/test - RetailRocket

In [13]:
train_df_retailrocket, test_df_retailrocket = safe_train_test_split(df_retailrocket)
print('RetailRocket train shape:', train_df_retailrocket.shape)
print('RetailRocket test shape:', test_df_retailrocket.shape)

RetailRocket train shape: (625215, 22)
RetailRocket test shape: (156304, 22)


## Dataset PyTorch avec contexte

Le dataset renvoie l'item, l'utilisateur, le contexte et le label.


In [14]:
class ContextMovielensDataset(Dataset):
    def __init__(self, df, context_columns):
        self.user_ids = torch.tensor(df['user_id_encoded'].values, dtype=torch.long)
        self.item_ids = torch.tensor(df['item_id_encoded'].values, dtype=torch.long)
        self.contexts = torch.tensor(df[context_columns].values, dtype=torch.float32)
        self.labels = torch.tensor(df['label'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.item_ids[idx], self.contexts[idx], self.labels[idx]

train_dataset = ContextMovielensDataset(train_df, context_cols)
test_dataset = ContextMovielensDataset(test_df, context_cols)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print('Train batches:', len(train_loader))
print('Test batches:', len(test_loader))


Train batches: 313
Test batches: 79


## Dataset PyTorch avec contexte - RetailRocket

In [15]:
train_dataset_retailrocket = ContextMovielensDataset(train_df_retailrocket, context_cols)
test_dataset_retailrocket = ContextMovielensDataset(test_df_retailrocket, context_cols)

batch_size_retailrocket = 256
train_loader_retailrocket = DataLoader(train_dataset_retailrocket, batch_size=batch_size_retailrocket, shuffle=True, num_workers=0)
test_loader_retailrocket = DataLoader(test_dataset_retailrocket, batch_size=batch_size_retailrocket, shuffle=False, num_workers=0)

print('RetailRocket Train batches:', len(train_loader_retailrocket))
print('RetailRocket Test batches:', len(test_loader_retailrocket))

RetailRocket Train batches: 2443
RetailRocket Test batches: 611


## Dataset PyTorch avec contexte - MovieLens 1M

In [16]:
train_dataset_ml_1m = ContextMovielensDataset(train_df_ml_1m, context_cols)
test_dataset_ml_1m = ContextMovielensDataset(test_df_ml_1m, context_cols)

batch_size_ml_1m = 256
train_loader_ml_1m = DataLoader(train_dataset_ml_1m, batch_size=batch_size_ml_1m, shuffle=True, num_workers=0)
test_loader_ml_1m = DataLoader(test_dataset_ml_1m, batch_size=batch_size_ml_1m, shuffle=False, num_workers=0)

print('ML-1M Train batches:', len(train_loader_ml_1m))
print('ML-1M Test batches:', len(test_loader_ml_1m))

ML-1M Train batches: 3124
ML-1M Test batches: 781


## Construction du modèle contextuel

Nous utilisons `NCFContext` avec une fusion par concaténation. Cet exemple peut être étendu ultérieurement avec `film` ou `attention`.


In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = NCFContext(
    num_users=n_users,
    num_items=n_items,
    context_dim=context_dim,
    embed_dim=32,
    mlp_layers=[64, 32, 16],
    context_embed_dim=32,
    dropout=0.2,
    fusion_type='concat'
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()

print(model)


Device: cpu
NCFContext(
  (ncf_core): NCF(
    (embedding_user_gmf): Embedding(943, 32)
    (embedding_item_gmf): Embedding(1682, 32)
    (embedding_user_mlp): Embedding(943, 32)
    (embedding_item_mlp): Embedding(1682, 32)
    (mlp): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=64, out_features=32, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.2, inplace=False)
      (6): Linear(in_features=32, out_features=16, bias=True)
      (7): ReLU()
      (8): Dropout(p=0.2, inplace=False)
    )
    (output_layer): Linear(in_features=48, out_features=1, bias=True)
  )
  (context_encoder): ContextEncoder(
    (net): Sequential(
      (0): Linear(in_features=8, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): Linear(in_features=64, out_features=32, bias=True)
      (4): ReLU()
    )
  )
  (final_mlp): Sequential(
    (0): Lin

## Construction du modèle contextuel - RetailRocket

In [18]:
model_retailrocket = NCFContext(
    num_users=n_users_retail,
    num_items=n_items_retail,
    context_dim=context_dim_retail,
    embed_dim=32,
    mlp_layers=[64, 32, 16],
    context_embed_dim=32,
    dropout=0.2,
    fusion_type='concat'
).to(device)
optimizer_retailrocket = torch.optim.Adam(model_retailrocket.parameters(), lr=1e-3)
print('RetailRocket Model created')

RetailRocket Model created


## Construction du modèle contextuel - MovieLens 1M

In [19]:
model_ml_1m = NCFContext(
    num_users=n_users_1m,
    num_items=n_items_1m,
    context_dim=context_dim_1m,
    embed_dim=32,
    mlp_layers=[64, 32, 16],
    context_embed_dim=32,
    dropout=0.2,
    fusion_type='concat'
).to(device)
optimizer_ml_1m = torch.optim.Adam(model_ml_1m.parameters(), lr=1e-3)
print('ML-1M Model created')

ML-1M Model created


## Entraînement

Nous entraînons le modèle avec une boucle simple et surveillons la perte d'entraînement.


In [20]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for user_batch, item_batch, context_batch, labels in loader:
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        context_batch = context_batch.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(user_batch, item_batch, context_batch)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

    return total_loss / len(loader.dataset)

epochs = 8
history = []
for epoch in range(1, epochs + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, device)
    history.append(loss)
    print(f'Epoch {epoch}/{epochs} - train loss: {loss:.4f}')


Epoch 1/8 - train loss: 0.6277
Epoch 2/8 - train loss: 0.5669
Epoch 3/8 - train loss: 0.5177
Epoch 4/8 - train loss: 0.3991
Epoch 5/8 - train loss: 0.2962
Epoch 6/8 - train loss: 0.2326
Epoch 7/8 - train loss: 0.1879
Epoch 8/8 - train loss: 0.1528


## Entraînement - RetailRocket

In [22]:
history_retailrocket = []
for epoch in range(1, epochs - 5):
    loss = train_epoch(model_retailrocket, train_loader_retailrocket, optimizer_retailrocket, criterion, device)
    history_retailrocket.append(loss)
    print(f'[RetailRocket] Epoch {epoch}/{epochs} - train loss: {loss:.4f}')

[RetailRocket] Epoch 1/8 - train loss: 0.0051
[RetailRocket] Epoch 2/8 - train loss: 0.0000


## Entraînement - MovieLens 1M

In [21]:
epochs = 8
history_ml_1m = []
for epoch in range(1, epochs + 1):
    loss = train_epoch(model_ml_1m, train_loader_ml_1m, optimizer_ml_1m, criterion, device)
    history_ml_1m.append(loss)
    print(f'[ML-1M] Epoch {epoch}/{epochs} - train loss: {loss:.4f}')

[ML-1M] Epoch 1/8 - train loss: 0.5631
[ML-1M] Epoch 2/8 - train loss: 0.4911
[ML-1M] Epoch 3/8 - train loss: 0.4183
[ML-1M] Epoch 4/8 - train loss: 0.3782
[ML-1M] Epoch 5/8 - train loss: 0.3528
[ML-1M] Epoch 6/8 - train loss: 0.3342
[ML-1M] Epoch 7/8 - train loss: 0.3194
[ML-1M] Epoch 8/8 - train loss: 0.3068


## Évaluation ranking

Nous évaluons sur un sous-ensemble de test pour maîtriser le temps d'inférence.


In [25]:
eval_df = test_df.sample(n=min(1000, len(test_df)), random_state=42).reset_index(drop=True)
all_item_ids = torch.arange(n_items, dtype=torch.long, device=device)

def evaluate_ranking(model, df, k=10):
    model.eval()
    hits, ndcgs, mrrs = [], [], []
    with torch.no_grad():
        for _, row in df.iterrows():
            user_id = torch.tensor([row['user_id_encoded']], dtype=torch.long, device=device)
            item_id = int(row['item_id_encoded'])
            context = torch.tensor(row[context_cols].values.astype(np.float32), dtype=torch.float32, device=device).unsqueeze(0)
            user_batch = user_id.expand(n_items)
            context_batch = context.expand(n_items, context_dim)
            logits = model(user_batch, all_item_ids, context_batch).cpu().numpy()
            ranking = np.argsort(-logits)
            hits.append(int(item_id in ranking[:k]))
            ndcgs.append(ndcg(ranking, item_id, k))
            mrrs.append(mrr(ranking, item_id))
    return np.mean(hits), np.mean(ndcgs), np.mean(mrrs)

hit10, ndcg10, mrr_score = evaluate_ranking(model, eval_df, k=10)
print(f'Hit Rate@10: {hit10:.4f}')
print(f'NDCG@10: {ndcg10:.4f}')
print(f'MRR: {mrr_score:.4f}')

Hit Rate@10: 0.0190
NDCG@10: 0.0098
MRR: 0.0124


## Évaluation ranking - RetailRocket

In [28]:
eval_df_retailrocket = test_df_retailrocket.sample(n=min(1000, len(test_df_retailrocket)), random_state=42).reset_index(drop=True)
hit10_retail, ndcg10_retail, mrr_retail = evaluate_ranking(model_retailrocket, eval_df_retailrocket, k=10)
print(f'[RetailRocket] Hit Rate@10: {hit10_retail:.4f}')
print(f'[RetailRocket] NDCG@10: {ndcg10_retail:.4f}')
print(f'[RetailRocket] MRR: {mrr_retail:.4f}')

[RetailRocket] Hit Rate@10: 0.0030
[RetailRocket] NDCG@10: 0.0009
[RetailRocket] MRR: 0.0006


## Évaluation ranking - MovieLens 1M

In [27]:
eval_df_ml_1m = test_df_ml_1m.sample(n=min(1000, len(test_df_ml_1m)), random_state=42).reset_index(drop=True)
hit10_ml_1m, ndcg10_ml_1m, mrr_ml_1m = evaluate_ranking(model_ml_1m, eval_df_ml_1m, k=10)
print(f'[ML-1M] Hit Rate@10: {hit10_ml_1m:.4f}')
print(f'[ML-1M] NDCG@10: {ndcg10_ml_1m:.4f}')
print(f'[ML-1M] MRR: {mrr_ml_1m:.4f}')

[ML-1M] Hit Rate@10: 0.0050
[ML-1M] NDCG@10: 0.0020
[ML-1M] MRR: 0.0038


## Sauvegarde du modèle contextuel

Nous sauvegardons les poids pour une comparaison avec le baseline.


In [29]:
RESULTS_DIR = (NOTEBOOK_ROOT / '..' / 'results' / 'models').resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = RESULTS_DIR / 'context_ncf_ml100k.pt'
torch.save(model.state_dict(), MODEL_PATH)
print('Saved context model to', MODEL_PATH)


Saved context model to /home/mrtds/Documents/my_projects/context-aware-recsys/results/models/context_ncf_ml100k.pt


## Sauvegarde des modèles contextuels - MovieLens 1M et RetailRocket

In [30]:
# Sauvegarder le modèle ML-1M contextuel
MODEL_PATH_ML_1M = RESULTS_DIR / 'context_ncf_ml1m.pt'
torch.save(model_ml_1m.state_dict(), MODEL_PATH_ML_1M)
print('Saved ML-1M context model to', MODEL_PATH_ML_1M)

# Sauvegarder le modèle RetailRocket contextuel
MODEL_PATH_RETAILROCKET = RESULTS_DIR / 'context_ncf_retailrocket.pt'
torch.save(model_retailrocket.state_dict(), MODEL_PATH_RETAILROCKET)
print('Saved RetailRocket context model to', MODEL_PATH_RETAILROCKET)

Saved ML-1M context model to /home/mrtds/Documents/my_projects/context-aware-recsys/results/models/context_ncf_ml1m.pt
Saved RetailRocket context model to /home/mrtds/Documents/my_projects/context-aware-recsys/results/models/context_ncf_retailrocket.pt


## Résumé

Ce notebook entraîne un NCF enrichi par des features contextuelles sur MovieLens 100K, MovieLens 1M et RetailRocket.
Nous comparons les performances des modèles contextuels sur ces trois jeux de données avec le baseline, en analysant l'apport des informations temporelles et de session dans la qualité du ranking.